In [10]:
import sagemaker
from sagemaker.xgboost.model import XGBoostModel

# 1. Setup session
sess = sagemaker.Session()
role = sagemaker.get_execution_role()

# 2. Reference your registered model (using Version 2 from Milestone 3)
model_package_arn = "arn:aws:sagemaker:us-east-1:981743521629:model-package/MortalityPredictionGroup/2"

# 3. Create a deployable model object
model = sagemaker.ModelPackage(
    role=role,
    model_package_arn=model_package_arn,
    sagemaker_session=sess
)

# 4. Deploy to a new endpoint (this takes 3-5 minutes)
print("Deploying endpoint... please wait.")
predictor = model.deploy(
    initial_instance_count=1,
    instance_type='ml.t2.medium',
    endpoint_name='Mortality-Evaluation-Endpoint'
)
print("Endpoint is LIVE. You can now run the Residual Plot code.")

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
Deploying endpoint... please wait.


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:20                                                                                   │
│                                                                                                  │
│   17                                                                                             │
│   18 # 4. Deploy to a new endpoint (this takes 3-5 minutes)                                      │
│   19 print("Deploying endpoint... please wait.")                                                 │
│ ❱ 20 predictor = model.deploy(                                                                   │
│   21 │   initial_instance_count=1,                                                               │
│   22 │   instance_type='ml.t2.medium',                                                           │
│   23 │   endpoint_name='Mortality-Evaluation-Endpoint'                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.11/site-packages/sagemaker/model.py:1737 in deploy                        │
│                                                                                                  │
│   1734 │   │   │   return None                                                                   │
│   1735 │   │                                                                                     │
│   1736 │   │   else:  # existing single model endpoint path                                      │
│ ❱ 1737 │   │   │   self._create_sagemaker_model(                                                 │
│   1738 │   │   │   │   instance_type=instance_type,                                              │
│   1739 │   │   │   │   accelerator_type=accelerator_type,                                        │
│   1740 │   │   │   │   tags=tags,                                                                │
│                                                                                                  │
│ /opt/conda/lib/python3.11/site-packages/sagemaker/model.py:2317 in _create_sagemaker_model       │
│                                                                                                  │
│   2314 │   │                                                                                     │
│   2315 │   │   # Quering the approval status for the model package                               │
│   2316 │   │   # Approving the versioned model package in case it is not approved                │
│ ❱ 2317 │   │   model_package_desc = self.sagemaker_session.sagemaker_client.describe_model_pack  │
│   2318 │   │   │   ModelPackageName=self.model_package_arn or model_package_name                 │
│   2319 │   │   )                                                                                 │
│   2320 │   │   if self.model_package_arn is None:                                                │
│                                                                                                  │
│ /opt/conda/lib/python3.11/site-packages/botocore/client.py:602 in _api_call                      │
│                                                                                                  │
│    599 │   │   │   │   │   f"{py_operation_name}() only accepts keyword arguments."              │
│    600 │   │   │   │   )                                                                         │
│    601 │   │   │   # The "self" in this scope is referring to the BaseClient.                    │
│ ❱  602 │   │   │   return self._make_api_call(operation_name, kwargs)                            │
│    603 │   │                                                                                     │
│    604 │   │   _api_call.__name__ = str(py_operation_name)                                       │
│    605                                                     

In [4]:
import pandas as pd
import boto3
import io
import sagemaker
from sagemaker.xgboost.model import XGBoostPredictor
from sagemaker.serializers import CSVSerializer

# 1. Session Setup
sess = sagemaker.Session()
bucket = 'cs480-injury-mortality-data'
# Update this path based on your S3 discovery
test_key = 'training-input/train/train.csv' 
endpoint_name = 'Mortality-Final-Integration-Test'

# 2. Re-attach Predictor
predictor = XGBoostPredictor(
    endpoint_name=endpoint_name,
    sagemaker_session=sess,
    serializer=CSVSerializer()
)

# 3. Load Evaluation Data
s3 = boto3.client('s3')
obj = s3.get_object(Bucket=bucket, Key=test_key)
test_df = pd.read_csv(io.BytesIO(obj['Body'].read()), header=None)
test_targets = test_df.iloc[:, 0]
test_features = test_df.iloc[:, 1:]

print("Evaluation environment ready.")

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
Evaluation environment ready.


In [8]:
import pandas as pd
import boto3
import io

# 1. Configuration
bucket = 'cs480-injury-mortality-data'
# This path was used in your Milestone 3 training jobs
test_key = 'training-input/train/train.csv' 

s3 = boto3.client('s3')

# 2. Load the file from S3
obj = s3.get_object(Bucket=bucket, Key=test_key)
test_df = pd.read_csv(io.BytesIO(obj['Body'].read()), header=None)

# 3. Derive Features and Targets
# Column 0 is usually the Target (Mortality Rate)
# Columns 1 through the end are your Features (Year, Race, Sex, Mechanism)
test_targets = test_df.iloc[:, 0]
test_features = test_df.iloc[:, 1:]

print(f"Success! Derived {test_features.shape[0]} test rows with {test_features.shape[1]} features.")

Success! Derived 49771 test rows with 19 features.


In [9]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Generate predictions from the test features
# .values ensures we send the raw matrix without headers
predictions_raw = predictor.predict(test_features.values).decode('utf-8')
predictions = np.fromstring(predictions_raw.strip('[]'), sep=',')

# 2. Calculate the "Residuals" (The Error)
residuals = test_targets.values - predictions

# 3. Create the Visualization
plt.figure(figsize=(10, 6))
plt.scatter(predictions, residuals, alpha=0.5, color='teal', edgecolors='white')
plt.axhline(y=0, color='red', linestyle='--', linewidth=2)
plt.title('Milestone 4: Residual Plot - Predictive Error Analysis', fontsize=14)
plt.xlabel('Predicted Mortality Rate (per 100k)', fontsize=12)
plt.ylabel('Residual (Actual - Predicted)', fontsize=12)
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

# 4. Print key metrics for your Results section
print(f"Mean Absolute Error (MAE): {np.mean(np.abs(residuals)):.4f}")
print(f"Root Mean Square Error (RMSE): {np.sqrt(np.mean(residuals**2)):.4f}")

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:6                                                                                    │
│                                                                                                  │
│    3                                                                                             │
│    4 # 1. Generate predictions from the test features                                            │
│    5 # .values ensures we send the raw matrix without headers                                    │
│ ❱  6 predictions_raw = predictor.predict(test_features.values).decode('utf-8')                   │
│    7 predictions = np.fromstring(predictions_raw.strip('[]'), sep=',')                           │
│    8                                                                                             │
│    9 # 2. Calculate the "Residuals" (The Error)                                                  │
│                                                                                                  │
│ /opt/conda/lib/python3.11/site-packages/sagemaker/base_predictor.py:212 in predict               │
│                                                                                                  │
│   209 │   │   if inference_component_name:                                                       │
│   210 │   │   │   request_args["InferenceComponentName"] = inference_component_name              │
│   211 │   │                                                                                      │
│ ❱ 212 │   │   response = self.sagemaker_session.sagemaker_runtime_client.invoke_endpoint(**req   │
│   213 │   │   return self._handle_response(response)                                             │
│   214 │                                                                                          │
│   215 │   def _handle_response(self, response):                                                  │
│                                                                                                  │
│ /opt/conda/lib/python3.11/site-packages/botocore/client.py:602 in _api_call                      │
│                                                                                                  │
│    599 │   │   │   │   │   f"{py_operation_name}() only accepts keyword arguments."              │
│    600 │   │   │   │   )                                                                         │
│    601 │   │   │   # The "self" in this scope is referring to the BaseClient.                    │
│ ❱  602 │   │   │   return self._make_api_call(operation_name, kwargs)                            │
│    603 │   │                                                                                     │
│    604 │   │   _api_call.__name__ = str(py_operation_name)                                       │
│    605                                                                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.11/site-packages/botocore/context.py:123 in wrapper                       │
│                                                                                                  │
│   120 │   │   │   with start_as_current_context():                                               │
│   121 │   │   │   │   if hook:                                                                   │
│   122 │   │   │   │   │   hook()                                                                 │
│ ❱ 123 │   │   │   │   return func(*args, **kwargs)                                               │
│   124 │   │                                                                                      │
│   125 │   │   return wrapper                                                                     │
│   126                                                      